# Assignment-search experiments: la_polyester & EDTA

Working notebook for `assignment_search.py`. Section 7 ("Module 1")
demonstrates it standalone against each case's real, solved donor
geometry.

**Update (2026-09): Module 2 is done.** `assignment_search.py` is now
wired into `engine.py`'s actual `solve()` loop as the `use_assignment_search`
parameter (`engine.py` 2.13.0, off by default -- confirmed byte-identical
on every case in this notebook with it off, and zero extra cost on
unambiguous cases like EDTA when it's on). Sections 1-8 below still run
every case through plain `engine.py` (flag off) unless a cell says
otherwise; **section 9** below demonstrates the flag turned on.

**la_polyester**: the `la_polyester_pent_bipyr` case registered in
section 2 uses a SMILES this session found does NOT encode the ester
chemistry its own note describes (section 6 has the RDKit atom-by-atom
proof) -- kept broken there on purpose so that discrepancy stays visible
rather than being silently papered over. Section 9 instead uses a
*corrected* SMILES the user supplied this session, and shows
`use_assignment_search` substantially improving its solve (best score
-16.76 -> -8.13) -- though it still does NOT fully converge; see that
section's own caveat before trusting it as chemically final.

**Update (2026-09, engine.py 2.14.0):** use_assignment_search got much
cheaper. Fe(H2O)6 (a fully symmetric case) was accidentally costing 4x
baseline (2.5s -> 11.2s) because the search kept generating several
physically-identical relabelings among chemically-interchangeable donors
and solving each one at full cost; it is now deduped to a single
candidate and runs at baseline speed (~2s). Genuinely ambiguous cases
like la_polyester now screen every candidate cheaply and only re-solve
the winner at full quality -- la_polyester dropped from an estimated
~58s (6 candidates x full solve) to ~17s. The tradeoff: cheap screening
can occasionally pick a different (slightly lower-scoring) winner than
an exhaustive full-quality search would -- section 9's score moved from
-6.94 to -8.13 for that reason. Still a big improvement over the -16.76
baseline, and still not converged -- see section 9.

## 0. Setup

In [ ]:
from __future__ import annotations

import math
import sys, os
import time
from pathlib import Path

os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

def _find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "jacobik").exists():
            return candidate
    raise RuntimeError(
        "Could not find the Jacobik project root (looked for pyproject.toml "
        "+ src/jacobik above the notebook's directory). Open this notebook "
        "from inside the project tree."
    )

PROJECT_ROOT = _find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from jacobik import ComplexBuilder, LigandInput
from jacobik.engine import AssignmentSearchConfig
from jacobik.stats.stats import StatsLookup, load_db
from jacobik.topology.coord_geom import classify as classify_geometry
from jacobik.topology.graph import build_heavy_ligand_graph
import networkx as nx

CONSTRAINTS_DB = PROJECT_ROOT / "src" / "jacobik" / "constraints.json"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Project root   : {PROJECT_ROOT}")
print(f"Constraints DB : {CONSTRAINTS_DB}  (exists={CONSTRAINTS_DB.exists()})")
print(f"Device         : {DEVICE}")


In [ ]:
import numpy as _np
import rdkit as _rdkit

print(f"python : {sys.version.split()[0]}")
print(f"torch  : {torch.__version__}")
print(f"numpy  : {_np.__version__}")
print(f"rdkit  : {_rdkit.__version__}")
print(f"cuda available : {torch.cuda.is_available()}")

STATS = StatsLookup(load_db(CONSTRAINTS_DB))
print("\nStats DB loaded.")


## 1. Realism-check helpers

Straight from `script_notebooks/la_polyester_test_case.ipynb` (verified
working this session) plus one new helper, `classify_binding_pattern`,
that buckets every donor pair by cis/trans *and* hop distance in one table
-- the direct way to see whether a case's converged geometry shows the
expected chelate pattern or a collapsed/wrong one.


In [ ]:
def donor_metal_distances(mol) -> tuple[list[int], np.ndarray]:
    donor_idxs = sorted(mol.donor_labels.keys())
    metal_pos = np.asarray(mol.get_atom(mol.metal_idx).position, dtype=float)
    dists = np.array(
        [np.linalg.norm(np.asarray(mol.get_atom(d).position, dtype=float) - metal_pos) for d in donor_idxs]
    )
    return donor_idxs, dists


def dmd_angles_deg(mol) -> tuple[list[int], np.ndarray, list[tuple[int, int]]]:
    donor_idxs, _ = donor_metal_distances(mol)
    metal_pos = np.asarray(mol.get_atom(mol.metal_idx).position, dtype=float)
    vecs = np.array([np.asarray(mol.get_atom(d).position, dtype=float) - metal_pos for d in donor_idxs])
    vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)
    n = len(vecs)
    angles, pairs = [], []
    for i in range(n):
        for j in range(i + 1, n):
            cos_t = float(np.clip(np.dot(vecs[i], vecs[j]), -1.0, 1.0))
            angles.append(math.degrees(math.acos(cos_t)))
            pairs.append((donor_idxs[i], donor_idxs[j]))
    return donor_idxs, np.array(angles), pairs


def find_chelate_pairs(mol, max_hops: int = 4) -> list[tuple[int, int, int]]:
    G = build_heavy_ligand_graph(mol.atoms, mol.bonds, mol.metal_idx)
    donor_idxs = sorted(mol.donor_labels.keys())
    pairs = []
    for i, di in enumerate(donor_idxs):
        if not G.has_node(di):
            continue
        for dj in donor_idxs[i + 1:]:
            if not G.has_node(dj):
                continue
            try:
                hops = nx.shortest_path_length(G, di, dj)
            except nx.NetworkXNoPath:
                continue
            if hops <= max_hops:
                pairs.append((di, dj, hops))
    return pairs


def all_hop_distances(mol) -> dict[tuple[int, int], int | None]:
    G = build_heavy_ligand_graph(mol.atoms, mol.bonds, mol.metal_idx)
    donor_idxs = sorted(mol.donor_labels.keys())
    out = {}
    for i, di in enumerate(donor_idxs):
        for dj in donor_idxs[i + 1:]:
            try:
                out[(di, dj)] = nx.shortest_path_length(G, di, dj)
            except nx.NetworkXNoPath:
                out[(di, dj)] = None
    return out


def classify_binding_pattern(mol) -> pd.DataFrame:
    # One row per donor pair: angle, cis/trans/other bucket, hop distance.
    # The fastest way to eyeball whether a converged geometry matches an
    # expected chelate pattern (e.g. EDTA's clean 12-cis/3-trans octahedron)
    # or looks collapsed/wrong (donors bunching where they shouldn't).
    donor_idxs, angles, pairs = dmd_angles_deg(mol)
    hops = all_hop_distances(mol)
    rows = []
    for (di, dj), ang in zip(pairs, angles):
        tag = "cis" if ang < 100 else ("trans" if ang > 150 else "other")
        rows.append({"donor_i": di, "donor_j": dj, "angle_deg": round(float(ang), 1),
                     "bucket": tag, "hops": hops.get((di, dj))})
    return pd.DataFrame(rows)


def check_shape_match(mol, expected_geometry: str) -> dict:
    actual = classify_geometry(mol)
    return {
        "pass": actual == expected_geometry,
        "expected": expected_geometry,
        "actual": actual,
        "message": "OK" if actual == expected_geometry else f"classified as {actual!r}, expected {expected_geometry!r}",
    }


def check_bond_length_spread(mol, max_spread_A: float = 0.35) -> dict:
    donor_idxs, dists = donor_metal_distances(mol)
    spread = float(dists.max() - dists.min()) if len(dists) else 0.0
    return {
        "pass": spread <= max_spread_A,
        "spread_A": spread,
        "min_A": float(dists.min()) if len(dists) else None,
        "max_A": float(dists.max()) if len(dists) else None,
        "message": "OK" if spread <= max_spread_A else f"M-D distances span {spread:.2f} A (max allowed {max_spread_A})",
    }


def check_dangling_arm(mol, threshold_A: float = 0.6) -> dict:
    donor_idxs, dists = donor_metal_distances(mol)
    if len(dists) == 0:
        return {"pass": True, "message": "no donors"}
    med = float(np.median(dists))
    excess = dists - med
    outlier_mask = excess > threshold_A
    outliers = [donor_idxs[i] for i in np.where(outlier_mask)[0]]
    return {
        "pass": not outlier_mask.any(),
        "median_A": med,
        "max_excess_A": float(excess.max()),
        "outlier_donors": outliers,
        "message": "OK" if not outliers else f"donor atom(s) {outliers} sit {float(excess.max()):.2f} A beyond median M-D distance",
    }


def check_metal_clashes(mol, min_A: float = 1.8) -> dict:
    donor_set = set(mol.donor_labels.keys())
    metal_pos = np.asarray(mol.get_atom(mol.metal_idx).position, dtype=float)
    dists = [
        float(np.linalg.norm(np.asarray(a.position, dtype=float) - metal_pos))
        for a in mol.atoms
        if a.idx != mol.metal_idx and a.idx not in donor_set and a.symbol != "H"
    ]
    closest = min(dists) if dists else float("inf")
    return {
        "pass": closest >= min_A,
        "closest_non_donor_A": closest,
        "message": "OK" if closest >= min_A else f"non-donor heavy atom {closest:.2f} A from metal (< {min_A})",
    }


def check_chelate_bite_angles(mol, lo: float = 60.0, hi: float = 115.0) -> dict:
    pairs = find_chelate_pairs(mol)
    if not pairs:
        return {"pass": True, "n_chelate_pairs": 0, "details": [], "message": "no chelate pairs"}
    metal_pos = np.asarray(mol.get_atom(mol.metal_idx).position, dtype=float)
    details = []
    for di, dj, hops in pairs:
        vi = np.asarray(mol.get_atom(di).position, dtype=float) - metal_pos
        vj = np.asarray(mol.get_atom(dj).position, dtype=float) - metal_pos
        cos_t = float(np.clip(np.dot(vi, vj) / (np.linalg.norm(vi) * np.linalg.norm(vj)), -1.0, 1.0))
        angle = math.degrees(math.acos(cos_t))
        details.append({"donors": (di, dj), "hops": hops, "bite_deg": angle, "pass": lo <= angle <= hi})
    passed = all(d["pass"] for d in details)
    worst = min(details, key=lambda d: min(abs(d["bite_deg"] - lo), abs(d["bite_deg"] - hi)) if not d["pass"] else 1e9)
    return {
        "pass": passed,
        "n_chelate_pairs": len(details),
        "details": details,
        "message": "OK" if passed else f"donors {worst['donors']} bite angle {worst['bite_deg']:.1f} deg outside [{lo},{hi}]",
    }


def pairwise_rmsd_matrix(conformers) -> np.ndarray:
    heavy_idx = [i for i, a in enumerate(conformers[0].complex.atoms) if a.symbol != "H" and a.idx != conformers[0].complex.metal_idx]
    coords = [np.array([c.complex.atoms[i].position for i in heavy_idx]) for c in conformers]
    n = len(conformers)
    M = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            Pc = coords[i] - coords[i].mean(axis=0)
            Qc = coords[j] - coords[j].mean(axis=0)
            H = Pc.T @ Qc
            U, _, Vt = np.linalg.svd(H)
            d = np.sign(np.linalg.det(Vt.T @ U.T))
            S = np.eye(3); S[-1, -1] = d
            R = Vt.T @ S @ U.T
            rmsd = float(np.sqrt(((Pc @ R.T - Qc) ** 2).sum(axis=-1).mean()))
            M[i, j] = M[j, i] = rmsd
    return M


def check_ensemble_diversity(conformers, collapsed_rmsd_A: float = 0.05) -> dict:
    if len(conformers) < 2:
        return {"pass": True, "message": "single conformer, diversity N/A"}
    M = pairwise_rmsd_matrix(conformers)
    iu = np.triu_indices_from(M, k=1)
    mean_offdiag = float(M[iu].mean())
    return {
        "pass": mean_offdiag > collapsed_rmsd_A,
        "mean_pairwise_rmsd_A": mean_offdiag,
        "message": "OK" if mean_offdiag > collapsed_rmsd_A else f"ensemble looks collapsed (mean pairwise RMSD {mean_offdiag:.3f} A)",
    }


def check_branch_symmetry(mol, tol_deg: float = 25.0) -> dict:
    """
    Regression check for the q3_mode_commit cross-ligand-instance bug
    (session notes: PI lanthanide/Fe aqua-nitrato task, fixed in
    engine.py 2.12.0 / ik_solver.py 2.8.0). Donor *placement* can be
    perfect (right site, right M-D distance, right D-M-D shape -- every
    other check above can pass) while the ligand's own local orientation
    around that donor is still chemically nonsensical: e.g. a coordinated
    water's two O-H bonds ending up wildly asymmetric relative to the
    M-O bond (M-O-H ~68 deg for one H, ~172 deg for the other, instead
    of the correct, symmetric ~120/120) because a since-fixed bug hard-
    committed chemically-independent monodentate ligands (separate water
    molecules, not one ligand's own multiple donor arms) to a single,
    sometimes-rare statistical mode. None of the checks above would ever
    catch this -- they all look at where DONOR atoms sit, not at where a
    donor's own non-donor substituents point.

    For every donor atom, finds its directly-bonded non-donor, non-metal
    substituent atoms (water O's two H's; an amine N's alkyl arms; etc.),
    groups them by element (only chemically-equivalent substituents are
    expected to be geometrically equivalent), and for each such group
    with >=2 members computes the M-donor-substituent angle for each
    member. Flags any group whose angles spread by more than tol_deg --
    real coordinated-ligand substituents on the same donor, symmetric by
    the ligand's own local chemistry, should land at very similar angles.
    """
    metal_pos = np.asarray(mol.get_atom(mol.metal_idx).position, dtype=float)
    donor_idxs = sorted(mol.donor_labels.keys())
    donor_set = set(donor_idxs)

    # Real covalent bonds only (mirrors Bond.is_coordination's < 0.75
    # cutoff for the opposite case -- a metal-donor contact is EXCLUDED
    # here via the metal_idx check below, not via this order filter, but
    # keeping it out too avoids picking up spurious weak/non-bonded
    # contacts as if they were substituents).
    adj: dict[int, list[int]] = {}
    for b in mol.bonds:
        if b.order < 0.75:
            continue
        adj.setdefault(b.i, []).append(b.j)
        adj.setdefault(b.j, []).append(b.i)

    details = []
    for d in donor_idxs:
        branch_idxs = [k for k in adj.get(d, []) if k != mol.metal_idx and k not in donor_set]
        if len(branch_idxs) < 2:
            continue
        d_pos = np.asarray(mol.get_atom(d).position, dtype=float)
        m_vec = metal_pos - d_pos
        by_elem: dict[str, list[int]] = {}
        for k in branch_idxs:
            by_elem.setdefault(mol.get_atom(k).symbol, []).append(k)
        for elem, ks in by_elem.items():
            if len(ks) < 2:
                continue
            angles = []
            for k in ks:
                b_vec = np.asarray(mol.get_atom(k).position, dtype=float) - d_pos
                cos_t = float(np.clip(
                    np.dot(m_vec, b_vec) / (np.linalg.norm(m_vec) * np.linalg.norm(b_vec)),
                    -1.0, 1.0,
                ))
                angles.append(math.degrees(math.acos(cos_t)))
            spread = max(angles) - min(angles)
            details.append({
                "donor": d, "branch_element": elem, "branches": ks,
                "M_donor_branch_angles_deg": [round(a, 1) for a in angles],
                "spread_deg": round(spread, 1),
                "pass": spread <= tol_deg,
            })

    passed = all(d["pass"] for d in details)
    worst = max(details, key=lambda d: d["spread_deg"]) if details else None
    return {
        "pass": passed,
        "n_checked": len(details),
        "details": details,
        "message": (
            "OK" if passed
            else "no symmetric branch groups found" if not details
            else f"donor {worst['donor']} ({worst['branch_element']} branches {worst['branches']}) "
                 f"M-donor-branch angles {worst['M_donor_branch_angles_deg']}, "
                 f"spread {worst['spread_deg']:.1f} deg (tol {tol_deg})"
        ),
    }


def run_all_checks(result, case: dict) -> dict:
    mol = result.best.complex
    return {
        "shape": check_shape_match(mol, case["geometry"]),
        "bond_spread": check_bond_length_spread(mol),
        "dangling_arm": check_dangling_arm(mol),
        "clashes": check_metal_clashes(mol),
        "chelate_bite": check_chelate_bite_angles(mol),
        "branch_symmetry": check_branch_symmetry(mol),
        "diversity": check_ensemble_diversity(result.conformers),
    }


## 2. Test case registry

In [ ]:
TEST_CASES = [
    dict(
        name="co_edta_octahedral",
        metal="Co", geometry="octahedral", charge=-1, spin=1,
        smiles="[O-]C(=O)CN(CCN(CC(=O)[O-])CC(=O)[O-])CC(=O)[O-]",
        donors=[[0], [4], [7], [11], [15], [19]], n_ligands=1,
        note="The flagship 'this one already works' baseline (per the "
             "user: la_polyester 'fails really badly despite seemingly "
             "being just like EDTA again'). Anionic hexadentate EDTA(4-), "
             "all four carboxylate donors the true single-bonded anionic "
             "oxygen (not a carbonyl mix -- see decompose.py 1.1.0/1.2.1). "
             "Same SMILES/donors as script_notebooks/reproduce_edta_results.py.",
    ),
    dict(
        name="fe_h2o_6_octahedral",
        metal="Fe", geometry="octahedral", charge=3, spin=6,
        smiles="O", donors=[[0]], n_ligands=6,
        note="[Fe(H2O)6]3+, high-spin d5 (mult=6). Simple, highly symmetric "
             "monodentate regression case for the q3_mode_commit cross-"
             "ligand-instance bug (fixed this session: engine.py 2.12.0 / "
             "ik_solver.py 2.8.0). symmetric_groups previously grouped all "
             "6 independent water ligands together (same SMILES + "
             "canonical rank) exactly like a single ligand's own multiple "
             "donor arms (e.g. EDTA's carboxylates), and q3_mode_commit "
             "hard-locked all 6 to one shared, sometimes-rare Q3 mode -- "
             "donor PLACEMENT was correct (right site, right shape) but "
             "each water's own internal orientation was chemically "
             "nonsensical (M-O-H ~68/172 deg instead of ~120/120), and it "
             "still passed shape/bond-length/clash checks because none of "
             "them look at a donor's own substituents. See "
             "check_branch_symmetry (section 1) for the check that catches "
             "this, and the PI lanthanide/Fe aqua-nitrato task report for "
             "the full root-cause writeup.",
    ),
    dict(
        name="la_polyester_pent_bipyr",
        metal="La", geometry="octahedral", charge=3, spin=1,
        smiles="C1C2COCC(=O)NCC3C=CC=C(COCC(=O)NCC(=CC=1)[C-]=2)[C-]=3",
        donors=[[3], [7], [15], [19], [24], [25]], n_ligands=1,
        note="*** SEE SECTION 6 BELOW BEFORE TRUSTING THIS CASE ***  "
             "Copied verbatim from script_notebooks/la_polyester_test_case.ipynb "
             "(the currently-registered SMILES/donors) so this notebook "
             "reproduces the same case that notebook runs. This session's "
             "investigation found that SMILES does NOT encode the ester "
             "chemistry its own note describes -- see section 6 for the "
             "RDKit atom-by-atom proof. Kept as-is here (not silently "
             "'fixed') so the discrepancy stays visible and reproducible "
             "rather than being papered over.",
    ),
]

CASES_BY_NAME = {c["name"]: c for c in TEST_CASES}
print(f"{len(TEST_CASES)} registered cases:")
for c in TEST_CASES:
    print(f"  - {c['name']}")


## 3. Run harness

In [ ]:
DEFAULT_SOLVE_KWARGS = dict(
    beam_width=16,
    n_ik_steps=200,
    lr=0.1,
    n_coord_samples=8,
    seed=42,
)


def build_case_complex(case: dict) -> ComplexBuilder:
    ligands = [LigandInput(case["smiles"], donors=case["donors"]) for _ in range(case["n_ligands"])]
    return ComplexBuilder(
        metal=case["metal"],
        geometry=case["geometry"],
        charge=case["charge"],
        spin=case["spin"],
        ligands=ligands,
    )


def run_case(case: dict, stats=STATS, debug: bool = False, **overrides):
    builder = build_case_complex(case)
    kwargs = {**DEFAULT_SOLVE_KWARGS, **case.get("solve_kwargs", {}), **overrides}

    t0 = time.perf_counter()
    result = builder.solve(constraints_db=CONSTRAINTS_DB, stats=stats, debug=debug, **kwargs)
    elapsed = time.perf_counter() - t0

    if result.best is not None:
        mol0 = result.best.complex
        subclasses = sorted({lbl.subclass for lbl in mol0.donor_labels.values()})
        print(
            f"[{case['name']}] solved in {elapsed:.1f}s -- "
            f"{len(result.converged)}/{len(result.conformers)} converged -- "
            f"best score={result.best.score:.4f} -- donor subclasses={subclasses}"
        )
    return result, elapsed


## 4. Batch run

In [ ]:
RESULTS: dict[str, object] = {}
ELAPSED: dict[str, float] = {}
CHECKS: dict[str, dict] = {}

rows = []
for case in TEST_CASES:
    try:
        result, elapsed = run_case(case, debug=False)
    except Exception as exc:
        rows.append({"case": case["name"], "error": str(exc)})
        print(f"[{case['name']}] FAILED TO SOLVE: {exc}")
        continue

    RESULTS[case["name"]] = result
    ELAPSED[case["name"]] = elapsed
    checks = run_all_checks(result, case)
    CHECKS[case["name"]] = checks

    row = {
        "case": case["name"],
        "n_converged": len(result.converged),
        "n_total": len(result.conformers),
        "best_score": result.best.score if result.best else float("nan"),
        "elapsed_s": round(elapsed, 1),
    }
    for check_name, check_result in checks.items():
        row[check_name] = check_result["pass"]
    rows.append(row)

SUMMARY_DF = pd.DataFrame(rows).set_index("case")
SUMMARY_DF


## 5. Per-case diagnostics

Donor subclasses, M-D distances, and the full pairwise D-M-D angle /
hop-distance table for each case's best conformer.


In [ ]:
for name, result in RESULTS.items():
    mol = result.best.complex
    donor_idxs, dists = donor_metal_distances(mol)
    print(f"=== {name} ===")
    print("donor atom indices (1-based):", donor_idxs)
    print("donor subclasses:", {d: mol.donor_labels[d].subclass for d in donor_idxs})
    print("M-D distances (A):", {d: round(float(v), 3) for d, v in zip(donor_idxs, dists)})
    print()
    df = classify_binding_pattern(mol)
    n_cis = (df["bucket"] == "cis").sum()
    n_trans = (df["bucket"] == "trans").sum()
    n_other = (df["bucket"] == "other").sum()
    print(f"D-M-D angle buckets: {n_cis} cis, {n_trans} trans, {n_other} other "
          f"(ideal octahedron with 6 unique donors: 12 cis @ ~90, 3 trans @ ~180, 0 other)")
    display(df)
    print()


## 6. Known issue: `la_polyester_pent_bipyr`'s SMILES doesn't match its own note

The registered SMILES was checked atom-by-atom with RDKit. The note (in
`script_notebooks/la_polyester_test_case.ipynb`, cell 6) describes donors
as "four ester ALKOXY oxygens ... plus two terminal alkoxide oxygens
([O-] termini)" forming a "malonate-diester unit". The actual SMILES
contains **no ester groups at all** (no `C(=O)O`) -- it has two amide
carbonyls (`C(=O)N`) and its two `[C-]` termini are anionic *aromatic
carbon* atoms (fulvenide/cyclopentadienide-like pi donors), not oxygens.
The six registered donor indices resolve to `O_ether_cyclic` x2,
`N_amide` x2, `C_pi` x2 -- not the `O_ether`/ester-alkoxy chemistry the
note (and the donor_classifier.py v1.2.0 fix earlier this session) was
about.

**Practical effect on this session's earlier finding**: the
`donor_classifier.py` v1.2.0 fix (ester-alkoxy-O vs carboxylate-O
misclassification) is still a real, independently-verified, correct fix
-- its 6 standalone unit tests pass on synthetic ester/carboxylate
fragments regardless of this case's SMILES. And the 0/16 to 15/16
convergence jump below is real and reproducible. But because this
molecule was never actually an ester in the first place, that jump can't
be attributed to the ester-classification fix specifically, and this
case's converged geometry (section 5 above) says nothing about the
2-cis/2-axial/2-cis binding-mode question from the uploaded structure --
it's a different molecule with different donor chemistry than intended.

**This needs a decision before la_polyester is useful as a test case
again**: either supply/confirm a corrected SMILES that actually encodes
the malonate-diester + ester-alkoxy chemistry the note describes, or
update the note to match what's actually being tested here.


In [ ]:
from rdkit import Chem

smiles = CASES_BY_NAME["la_polyester_pent_bipyr"]["smiles"]
donor_atom_idxs_0based = [d[0] for d in CASES_BY_NAME["la_polyester_pent_bipyr"]["donors"]]

mol_check = Chem.MolFromSmiles(smiles, sanitize=False)
Chem.SanitizeMol(mol_check, sanitizeOps=Chem.SANITIZE_ALL ^ Chem.SANITIZE_KEKULIZE, catchErrors=True)
print(f"SMILES: {smiles}")
print(f"registered donor atom indices (0-based, RDKit numbering): {donor_atom_idxs_0based}\n")
for i, atom in enumerate(mol_check.GetAtoms()):
    marker = "  <-- DONOR" if i in donor_atom_idxs_0based else ""
    nbrs = [n.GetSymbol() + str(n.GetIdx()) for n in atom.GetNeighbors()]
    print(f"{i:3d} {atom.GetSymbol():3s} charge={atom.GetFormalCharge():+d} aromatic={atom.GetIsAromatic()!s:5} neighbors={nbrs}{marker}")


## 7. Module 1 demo: `assignment_search.py` on real, solved donor geometry

`assignment_search.py` generates a small, diversity-capped set of
alternative donor-to-site assignments around the baseline one-shot
Hungarian solve, for cases where that single arbitrary initial embedding
may have locked onto the wrong topology. **It is not yet wired into
`engine.py`'s solve loop** (that's the pending engine-integration work) --
this section runs it standalone against each case's already-solved donor
geometry, purely to sanity-check its behavior on real molecules rather
than only the synthetic adversarial test used during this session's
development.

Expected behavior: on EDTA's clean, already-unambiguous geometry, it
should return just the baseline (no wasted alternatives). On
la_polyester's messier, distorted geometry (see section 6 for why that
distortion may not be chemically meaningful) it may propose several
alternatives -- worth inspecting, but not a verdict on the ester
binding-mode question until the SMILES issue above is resolved.


In [ ]:
import importlib.util

_spec = importlib.util.spec_from_file_location(
    "jacobik.ik.assignment_search", PROJECT_ROOT / "src" / "jacobik" / "ik" / "assignment_search.py"
)
assignment_search = importlib.util.module_from_spec(_spec)
sys.modules["jacobik.ik.assignment_search"] = assignment_search
_spec.loader.exec_module(assignment_search)

from jacobik.ik import targets as targets_mod

IDEAL_OCTAHEDRAL = targets_mod.IDEAL_GEOMETRIES["octahedral"]


def extract_donor_geometry(mol):
    # Real (donor_directions, hop_matrix, donor_subclasses, chelate_pairs)
    # for a solved complex -- the same inputs generate_candidate_assignments
    # would need if wired into engine.py's assignment step.
    donor_idxs = sorted(mol.donor_labels.keys())
    metal_pos = np.asarray(mol.get_atom(mol.metal_idx).position, dtype=float)
    donor_dirs = np.array(
        [np.asarray(mol.get_atom(d).position, dtype=float) - metal_pos for d in donor_idxs]
    )
    donor_dirs = donor_dirs / np.linalg.norm(donor_dirs, axis=1, keepdims=True)

    G = build_heavy_ligand_graph(mol.atoms, mol.bonds, mol.metal_idx)
    n = len(donor_idxs)
    hop_matrix = np.full((n, n), -1, dtype=int)
    for i, di in enumerate(donor_idxs):
        for j, dj in enumerate(donor_idxs):
            if i == j:
                continue
            try:
                hop_matrix[i, j] = nx.shortest_path_length(G, di, dj)
            except nx.NetworkXNoPath:
                pass

    chelate_pairs = [(i, j) for i in range(n) for j in range(i + 1, n) if 0 <= hop_matrix[i, j] <= 4]
    donor_subclasses = [mol.donor_labels[d].subclass for d in donor_idxs]
    return donor_dirs, hop_matrix, donor_subclasses, chelate_pairs, donor_idxs


for name, result in RESULTS.items():
    mol = result.best.complex
    donor_dirs, hop_matrix, donor_subclasses, chelate_pairs, donor_idxs = extract_donor_geometry(mol)

    candidates = assignment_search.generate_candidate_assignments(
        donor_dirs, IDEAL_OCTAHEDRAL,
        donor_subclasses=donor_subclasses, geometry_label="octahedral",
        chelate_pairs=chelate_pairs, hop_matrix=hop_matrix,
    )
    print(f"=== {name} ===")
    print(f"donor atom indices: {donor_idxs}")
    print(f"{len(candidates)} candidate assignment(s) generated (index 0 = baseline Hungarian solve)")
    site_names = ["+x", "-x", "+y", "-y", "+z", "-z"]
    for i, c in enumerate(candidates):
        tag = "(baseline)" if i == 0 else f"(alt {i})"
        print(f"  {tag} {[site_names[s] for s in c]}")
    print()


## 8. 3D structure

In [ ]:
_ELEMENT_COLORS = {
    "H": "#dddddd", "C": "#4d4d4d", "N": "#3050f8", "O": "#ff0d0d",
    "P": "#ff8000", "S": "#ffff30", "Cl": "#1ff01f", "F": "#90e050",
}
_METAL_COLOR = "#b8860b"


def plot_structure_3d(mol, title="", show_hydrogen=False, elev=20, azim=35):
    fig = plt.figure(figsize=(5, 5))
    ax = fig.add_subplot(111, projection="3d")

    donor_set = set(mol.donor_labels.keys())
    for a in mol.atoms:
        if a.symbol == "H" and not show_hydrogen:
            continue
        color = _METAL_COLOR if a.idx == mol.metal_idx else _ELEMENT_COLORS.get(a.symbol, "#999999")
        size = 260 if a.idx == mol.metal_idx else (120 if a.idx in donor_set else 55)
        edge = "black" if a.idx in donor_set or a.idx == mol.metal_idx else "none"
        ax.scatter(*a.position, color=color, s=size, edgecolors=edge, linewidths=1.0, depthshade=True)

    atom_map = {a.idx: a for a in mol.atoms}
    for b in mol.bonds:
        ai, aj = atom_map.get(b.i), atom_map.get(b.j)
        if ai is None or aj is None:
            continue
        if not show_hydrogen and (ai.symbol == "H" or aj.symbol == "H"):
            continue
        style = "--" if (b.i == mol.metal_idx or b.j == mol.metal_idx) else "-"
        lw = 1.2 if style == "--" else 1.6
        ax.plot(*zip(ai.position, aj.position), style, color="#555555", linewidth=lw)

    ax.set_title(title)
    ax.set_box_aspect((1, 1, 1))
    ax.view_init(elev=elev, azim=azim)
    ax.set_axis_off()
    plt.tight_layout()
    plt.show()


for name, result in RESULTS.items():
    plot_structure_3d(result.best.complex, f"{name} (best)")


## 8b. Interactive 3D structure (py3Dmol)

Same best conformer as above, rendered with the project's own
`jacobik.vis.stats_view` helpers (py3Dmol/3Dmol.js) instead of the
dependency-free matplotlib fallback -- draggable/zoomable, with donor
bonds and labels drawn the same way `Visualizer.view_ensemble_structs`
does. Requires `py3Dmol` + `ipywidgets` in the kernel (already in this
project's `benchmark` extras / this session's environment).


In [ ]:
from jacobik.vis.stats_view import _make_view, _render_complex, _finalise

def show_structure_3d(mol, width=650, height=500, show_hydrogens=False):
    v = _make_view(width, height)
    _render_complex(v, mol, show_hydrogens=show_hydrogens)
    _finalise(v)
    return v.show()

for name, result in RESULTS.items():
    print(f"=== {name} ===")
    show_structure_3d(result.best.complex)


## 9. Module 2 live: la_polyester (corrected SMILES, 2026-09)

`assignment_search.py` ("Module 1", section 7 above) is now wired into
`engine.py`'s actual `solve()` loop, as `engine.py` 2.13.0's
`use_assignment_search` parameter (default `False`). This section runs
it end-to-end on a *corrected* la_polyester SMILES the user supplied this
session -- real alkoxide (`[O-]`) termini instead of the carbanion
(`[C-]`) SMILES bug documented in section 6 / `la_polyester_pent_bipyr`
above, which is left broken there on purpose so that discrepancy stays
visible and reproducible.

Not added to `TEST_CASES` above: this case does not converge by the
solver's strict internal criterion (a genuinely long, floppy,
low-symmetry ligand -- consistent with `la_polyester_pent_bipyr` already
being flagged unreliable in section 6), so it would just read as a
permanent "fail" row in `SUMMARY_DF`. Run standalone here instead, so the
before/after comparison stays legible on its own terms.

In [ ]:
LA_POLYESTER_V2_SMILES = "C(=O)(OCC(=O)OC(C)C(C)[O-])CC(=O)OCC(C)OC(=O)C[O-]"
LA_POLYESTER_V2_DONORS = [[2], [6], [11], [15], [19], [23]]

# Sanity-check the SMILES/donor indices before spending solve time on them.
from rdkit import Chem as _Chem2

_mol2 = _Chem2.MolFromSmiles(LA_POLYESTER_V2_SMILES)
assert _mol2 is not None, "RDKit failed to parse LA_POLYESTER_V2_SMILES"
_donor_syms2 = [_mol2.GetAtomWithIdx(d[0]).GetSymbol() for d in LA_POLYESTER_V2_DONORS]
assert all(s == "O" for s in _donor_syms2), f"Not all donors are O: {_donor_syms2}"
_ligand_charge2 = sum(a.GetFormalCharge() for a in _mol2.GetAtoms())
LA_POLYESTER_V2_CHARGE = 3 + _ligand_charge2  # La3+ + ligand's own formal charge (two [O-] termini -> -2)
print(f"Donors are all O: {_donor_syms2}")
print(f"Ligand formal charge: {_ligand_charge2:+d}  ->  complex charge: {LA_POLYESTER_V2_CHARGE:+d} (La3+)")

_la2_ligand = LigandInput(LA_POLYESTER_V2_SMILES, donors=LA_POLYESTER_V2_DONORS)
_la2_builder_kwargs = dict(metal="La", geometry="octahedral", charge=LA_POLYESTER_V2_CHARGE, spin=1)
_la2_solve_kwargs = dict(constraints_db=CONSTRAINTS_DB, stats=STATS, seed=0, beam_width=32, n_ik_steps=200)

print("\n=== baseline (use_assignment_search=False) ===")
t0 = time.perf_counter()
LA2_RESULT_BASELINE = ComplexBuilder(ligands=[_la2_ligand], **_la2_builder_kwargs).solve(
    use_assignment_search=False, **_la2_solve_kwargs
)
_dt_baseline = time.perf_counter() - t0
print(f"  {len(LA2_RESULT_BASELINE.converged)}/{len(LA2_RESULT_BASELINE.conformers)} converged -- "
      f"best score={LA2_RESULT_BASELINE.best.score:.4f} -- {_dt_baseline:.1f}s")

print("\n=== assignment search on (use_assignment_search=True, k_max=6) ===")
t0 = time.perf_counter()
LA2_RESULT_SEARCH = ComplexBuilder(ligands=[_la2_ligand], **_la2_builder_kwargs).solve(
    use_assignment_search=True, assignment_search=AssignmentSearchConfig(k_max=6),
    **_la2_solve_kwargs
)
_dt_search = time.perf_counter() - t0
print(f"  {len(LA2_RESULT_SEARCH.converged)}/{len(LA2_RESULT_SEARCH.conformers)} converged -- "
      f"best score={LA2_RESULT_SEARCH.best.score:.4f} -- {_dt_search:.1f}s")
print(f"  assignment_search info: {LA2_RESULT_SEARCH.assignment_search}")

print(f"\nScore improvement from assignment search: "
      f"{LA2_RESULT_SEARCH.best.score - LA2_RESULT_BASELINE.best.score:+.4f}")

In [ ]:
for _label, _res in [("baseline", LA2_RESULT_BASELINE), ("assignment search", LA2_RESULT_SEARCH)]:
    _idxs, _dists = donor_metal_distances(_res.best.complex)
    print(f"{_label:18s}: M-O distances = " + ", ".join(f"{d:.3f}" for d in _dists) +
          f"  (range {_dists.min():.3f}-{_dists.max():.3f} A)")

In [ ]:
plot_structure_3d(LA2_RESULT_BASELINE.best.complex, "la_polyester (corrected SMILES) -- baseline")
plot_structure_3d(LA2_RESULT_SEARCH.best.complex, "la_polyester (corrected SMILES) -- assignment search")

In [ ]:
print("Interactive 3D view of the assignment-search winner:")
show_structure_3d(LA2_RESULT_SEARCH.best.complex)

## 10. Debugging playground

Pick any registered case, rerun with `debug=True` for full solver
diagnostics, and get the score breakdown / structure in one place. Edit
`CASE_NAME` and any `overrides` and rerun.


In [ ]:
CASE_NAME = "co_edta_octahedral"  # or "fe_h2o_6_octahedral" / "la_polyester_pent_bipyr"
case = CASES_BY_NAME[CASE_NAME]

result, elapsed = run_case(case, debug=True)  # add overrides as kwargs, e.g. seed=7, beam_width=32
checks = run_all_checks(result, case)

print("\n=== checks ===")
for k, v in checks.items():
    print(f"  {k:14s} pass={v['pass']!s:5}  {v.get('message', '')}")

def score_breakdown_for(result, conformer=None) -> dict[str, float]:
    conformer = conformer or result.best
    X = torch.tensor(
        np.stack([a.position for a in conformer.complex.atoms]), dtype=torch.float32
    ).unsqueeze(0)
    _, breakdown = result.scorer.score(X)
    return {k: float(v.item()) for k, v in breakdown.items()}

print("\n=== score breakdown ===")
for k, v in score_breakdown_for(result).items():
    print(f"  {k}: {v:.4f}")

plot_structure_3d(result.best.complex, f"{CASE_NAME} (best)")
